# MCP Supply-Chain Risk Management

This guided lab admits a critical third-party MCP server only when supplier and artifact claims form a current, exact, authenticated evidence graph. It is offline and does not contact a vendor, registry, scanner, or production system.

## Learning contract

You will map a rooted dependency graph, distinguish supplier assurance from artifact assurance, verify exact evidence bindings, fail closed on gaps and findings, authorize a decision, exercise a bounded exception, invalidate admission after compromise, and report portfolio coverage.

In [ ]:
from dataclasses import replace
from pathlib import Path
from pprint import pprint
import runpy

lab = runpy.run_path(Path('lab.py'))
NOW = 1_000
principals, authority, engine, approvals, registry, request = lab['demo_components'](NOW)
evidence = lab['evidence_fixture'](authority, request, NOW)
print(f'Loaded {len(request.components)} components and {len(evidence)} signed evidence artifacts.')

## 1. Start with business authority and dependency reach

The vendor MCP server can read and update confidential support tickets. Its root artifact depends on an SDK. A registry listing, supplier name, or package tag cannot safely join these facts; the intake uses stable IDs, immutable versions, and digests.

In [ ]:
pprint({
    'supplier': request.supplier.supplier_id,
    'tier': request.supplier.tier.value,
    'tenant': request.tenant_scope,
    'use_case': request.use_case,
    'exit_plan': request.supplier.exit_plan,
})
for component in request.components:
    pprint(component.payload())

## 2. Validate a rooted, reachable component graph

The application validates ownership, profile freshness, one root, unique components, exact supplier binding, immutable digests, PURLs, declared capabilities, and graph reachability.

In [ ]:
engine.validate_intake(request, NOW)
root = next(item for item in request.components if item.root)
assert set(root.dependency_ids) == {'component:sdk'}
print('root:', root.component_id, 'dependencies:', root.dependency_ids)

In [ ]:
orphan_root = replace(root, dependency_ids=())
try:
    engine.validate_intake(
        replace(request, components=(orphan_root, request.components[1])), NOW
    )
    raise AssertionError('orphan inventory must fail')
except lab['SupplyChainError'] as exc:
    print(exc.code)
    assert exc.code == 'DEPENDENCY_GRAPH_INCOMPLETE'

## 3. Keep supplier and artifact evidence separate

Critical supplier evidence covers due diligence, security program, contract, response, continuity, data flow, and independent assessment. Each component separately needs signature, provenance, SBOM, vulnerability evidence, and kind-specific evidence such as license review.

In [ ]:
supplier_required = engine.policy.supplier_requirements[request.supplier.tier]
print('supplier evidence:', sorted(item.value for item in supplier_required))
for component in request.components:
    required = engine.policy.component_requirements[component.kind]
    print(component.component_id, sorted(item.value for item in required))

In [ ]:
verified = [engine.validate_evidence(item, request, NOW + 1) for item in evidence]
for item in verified:
    print(item.evidence_id, item.subject_type, item.subject_id, item.kind.value)

### Adversarial check: signatures authenticate exact content

Changing a signed numerator must break verification. Even an authentic passing claim must match the exact supplier profile or component version and digest.

In [ ]:
tampered = replace(evidence[0], artifact=replace(evidence[0].artifact, numerator=0))
try:
    engine.validate_evidence(tampered, request, NOW + 1)
    raise AssertionError('tampered evidence must fail')
except lab['SupplyChainError'] as exc:
    print(exc.code)
    assert exc.code == 'EVIDENCE_SIGNATURE_INVALID'

In [ ]:
component_evidence = next(
    item for item in evidence if item.artifact.subject_type == 'component'
)
wrong_subject = authority.sign(
    replace(component_evidence.artifact, subject_version='old')
)
try:
    engine.validate_evidence(wrong_subject, request, NOW + 1)
    raise AssertionError('evidence for another version must fail')
except lab['SupplyChainError'] as exc:
    print(exc.code)
    assert exc.code == 'EVIDENCE_SUBJECT_MISMATCH'

## 4. Make assurance gaps visible

Remove one supplier assessment and one SDK SBOM. The result names both gaps; no score or model summary can hide them.

In [ ]:
gapped = tuple(
    item for item in evidence
    if item.artifact.evidence_id not in {
        'EV-SUP-DUE_DILIGENCE', 'EV-SDK-SBOM'
    }
)
gap = engine.evaluate(request, gapped, (), (), NOW + 1)
pprint(gap.blockers)
assert 'supplier:missing:due_diligence' in gap.blockers
assert 'component:component:sdk:missing:sbom' in gap.blockers

## 5. Provenance does not erase a vulnerability

All artifact evidence can be green while an open critical affected finding still blocks admission. Vulnerability state is a separate input.

In [ ]:
critical = lab['Finding'](
    'CVE-CRITICAL', 'component:sdk', lab['FindingSeverity'].CRITICAL,
    lab['FindingStatus'].OPEN, lab['Exploitability'].AFFECTED,
    'OSV', 'osv-db-2026-10-30', NOW, NOW + 86_400,
)
denied = engine.propose_decision(
    request, evidence, (critical,), (), NOW + 1, 'DEC-NB-DENY',
)
print(denied.recommendation.value, denied.blockers)
assert denied.recommendation is lab['Decision'].DENY

## 6. Exercise a bounded high-finding exception

The exception binds one finding, component, version, digest, and tenant. It requires a separately approved receipt plus component-bound compensating-control evidence, monitoring, review, expiry, and exit.

In [ ]:
_, authority2, engine2, approvals2, registry2, request2 = lab['demo_components'](NOW)
evidence2 = lab['evidence_fixture'](authority2, request2, NOW)
sdk = next(item for item in request2.components if item.component_id == 'component:sdk')
high = lab['Finding'](
    'OSV-2026-001', sdk.component_id, lab['FindingSeverity'].HIGH,
    lab['FindingStatus'].OPEN, lab['Exploitability'].AFFECTED,
    'OSV', 'osv-db-2026-10-30', NOW, NOW + 86_400,
)
control_artifact = lab['EvidenceArtifact'](
    'EV-COMPENSATION', lab['EvidenceKind'].COMPENSATING_CONTROL_TEST,
    'component', sdk.component_id, sdk.version, sdk.artifact_digest,
    'producer:ci', True, lab['EvidenceOutcome'].PASS,
    'network deny policy blocks the vulnerable path',
    NOW, NOW + 86_400, 'f' * 64,
    '25 labelled exploit attempts', 25, 25, 'attempts blocked',
)
control_evidence = authority2.sign(control_artifact)

In [ ]:
exception = lab['ExceptionRequest'](
    'EX-NB-001', high.finding_id, sdk.component_id, sdk.version,
    sdk.artifact_digest, request2.tenant_scope, request2.business_owner_id,
    'temporary exposure while fixed SDK is qualified',
    ('network-deny-vulnerable-path',), ('EV-COMPENSATION',),
    'alert on attempted vulnerable operations',
    'upgrade to fixed SDK and remove the exception',
    NOW, NOW + 2 * 86_400, NOW + 5 * 86_400,
)
exception_receipt = approvals2.issue(
    'supply_chain_exception', exception.payload(), request2.business_owner_id,
    'approver:ciso', 'ciso', NOW,
)
active = registry2.activate_exception(
    exception, exception_receipt.receipt_id, request2.business_owner_id, NOW + 1,
)
conditional = engine2.propose_decision(
    request2, evidence2 + (control_evidence,), (high,), (active,),
    NOW + 2, 'DEC-NB-CONDITIONAL',
)
print(conditional.recommendation.value, conditional.exception_ids)
assert conditional.recommendation is lab['Decision'].CONDITIONAL

## 7. Authorize the exact critical-supplier decision

Critical supplier policy requires the supply-chain committee. Approval binds the complete proposal digest and is expiring and atomically single-use.

In [ ]:
proposal = engine.propose_decision(
    request, evidence, (), (), NOW + 1, 'DEC-NB-APPROVE',
)
receipt = approvals.issue(
    'supply_chain_decision', proposal.payload(), request.business_owner_id,
    'committee:1', 'supply-chain-committee', NOW + 1,
)
admitted = registry.apply_decision(
    proposal, receipt.receipt_id, request.business_owner_id,
    'operation-notebook-001', NOW + 2,
)
print(admitted.state.value, admitted.version, admitted.decision_id)
assert admitted.state is lab['AdmissionState'].APPROVED

## 8. Invalidate admission after material change

A signer compromise is not a request to update a dashboard. It clears the prior decision and evidence set and moves the record to quarantine.

In [ ]:
quarantined = registry.apply_signal(
    request.intake_id, 'signal-signer-001',
    lab['SignalKind'].SIGNER_COMPROMISE,
    request.supplier.profile_version, 'signer-revoked-v1', NOW + 3,
)
print(quarantined.state.value, quarantined.decision_id, quarantined.verified_evidence_ids)
assert quarantined.state is lab['AdmissionState'].QUARANTINED
assert quarantined.decision_id is None

## 9. Report the portfolio with explicit denominators

Runtime discovery observes a second supplier that is not in the governed register. Coverage reports 1/2, rather than silently defining the register as the complete population.

In [ ]:
metrics = lab['portfolio_metrics'](
    registry.records.values(),
    ('supplier:helpdesk', 'supplier:shadow'),
    registry.exceptions.values(),
    NOW + 3,
)
pprint(metrics)
assert metrics['supplier_inventory_coverage'] == 0.5
assert metrics['quarantined_record_count'] == 1

## 10. Gate offboarding after invalidation

The lab records an offboarding terminal transition only after denial, reassessment, or quarantine. Production must separately verify credential removal, data handling, catalog removal, replacement readiness, and forbidden-effect absence.

In [ ]:
offboarded = registry.offboard(
    request.intake_id, 'operation-offboard-001', NOW + 4,
)
print(offboarded.state.value)
assert offboarded.state is lab['AdmissionState'].OFFBOARDED

In [ ]:
for event in registry.audit:
    pprint(event)
assert [event.event_type for event in registry.audit] == [
    'supply_chain.decision', 'supply_chain.signal', 'supply_chain.offboard'
]

## Production boundary

Replace fixture keys, evidence, inventory, and journals with authenticated suppliers and producers, real Cosign/in-toto/SLSA and SBOM verification, current scanner/VEX inputs, durable transactional stores, registry/runtime reconciliation, policy-as-code, protected audit, kill switches, and outcome-verified revocation and exit.

## Reflection

1. Which suppliers or components can change MCP authority without changing your application repository?
2. How would you discover a fourth-party service omitted from the supplier questionnaire?
3. Which evidence proves an SBOM is complete for the deployed artifact?
4. Which signals require reassessment, and which require immediate quarantine?
5. How would you verify that offboarding removed every old credential and effect path?
6. What named population and denominator should your inventory-coverage metric use?